In [1]:

# !pip install transformers


In [2]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.nn.functional import softmax
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

In [3]:
# !unzip train_essays.csv.zip

Archive:  train_essays.csv.zip
  inflating: train_essays.csv        


In [4]:
# !unzip train_drcat_02.csv.zip

Archive:  train_drcat_02.csv.zip
  inflating: train_drcat_02.csv      


In [5]:
# !unzip bert.zip

Archive:  bert.zip
  inflating: config.json             
  inflating: pytorch_model.bin       
  inflating: vocab.txt               


In [6]:
train_df = pd.read_csv('train_essays.csv')

train_df

,id,prompt_id,text,generated
0,0059830c,0,Cars. Cars have been around since they became ...,0
1,005db917,0,Transportation is a large necessity in most co...,0
2,008f63e3,0,"""America's love affair with it's vehicles seem...",0
3,00940276,0,How often do you ride in a car? Do you drive a...,0
4,00c39458,0,Cars are a wonderful thing. They are perhaps o...,0
...,...,...,...,...
1373,fe6ff9a5,1,There has been a fuss about the Elector Colleg...,0
1374,ff669174,0,Limiting car usage has many advantages. Such a...,0
1375,ffa247e0,0,There's a new trend that has been developing f...,0
1376,ffc237e9,0,As we all know cars are a big part of our soci...,0


In [7]:
train_df.rename(columns={'generated': 'label'}, inplace=True)

In [8]:
train_df['label'].value_counts()

0    1375
1       3
Name: label, dtype: int64

In [9]:
train_df.drop(['id', 'prompt_id'], axis=1, inplace=True)
train_df

,text,label
0,Cars. Cars have been around since they became ...,0
1,Transportation is a large necessity in most co...,0
2,"""America's love affair with it's vehicles seem...",0
3,How often do you ride in a car? Do you drive a...,0
4,Cars are a wonderful thing. They are perhaps o...,0
...,...,...
1373,There has been a fuss about the Elector Colleg...,0
1374,Limiting car usage has many advantages. Such a...,0
1375,There's a new trend that has been developing f...,0
1376,As we all know cars are a big part of our soci...,0


In [10]:
train = pd.read_csv("train_drcat_02.csv")
train['label'].value_counts()
train.drop(['essay_id', 'source', 'prompt', 'fold'], axis=1, inplace=True)
# train
# train1 = train[train.RDizzl3_seven == False].reset_index(drop=True)
train1=train[train["label"]==1].sample(1300)
# train = train[train.RDizzl3_seven == True].reset_index(drop=True)

train_df=pd.concat([train_df,train1])

# df = df.drop(['prompt_name','source','RDizzl3_seven'],axis = 1)

In [11]:
train_df = train_df.sample(frac=1, random_state=1)
train_df.reset_index(drop=True, inplace=True)

split_index_1 = int(len(train_df) * 0.7)
split_index_2 = int(len(train_df) * 0.85)

train_df, val_df, test_df = train_df[:split_index_1], train_df[split_index_1:split_index_2], train_df[split_index_2:]

len(train_df), len(val_df), len(test_df)

(1874, 402, 402)

In [15]:
# Tokenize the data
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = str(self.data.iloc[idx]['text'])
        label = int(self.data.iloc[idx]['label'])

        encoding = self.tokenizer(
             text,
             max_length=self.max_length,  # Set the maximum length of the sequence
             add_special_tokens=True,  # Add [CLS] and [SEP] tokens
             return_token_type_ids=False,
             pad_to_max_length=True,
             return_attention_mask=True,
             return_tensors='pt'  # Return PyTorch tensors
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Create datasets and dataloaders
train_dataset = TextDataset(train_df, tokenizer, max_length=128)
val_dataset = TextDataset(val_df, tokenizer, max_length=128)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# Load pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Define optimizer and learning rate scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
num_epochs = 10
model.to('cuda')

# Training loop
for epoch in range(num_epochs):
    model.train()

    for batch in train_loader:
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['label'].to('cuda')

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_losses = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to('cuda')
            attention_mask = batch['attention_mask'].to('cuda')
            labels = batch['label'].to('cuda')

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            val_losses.append(outputs.loss.item())

    avg_val_loss = sum(val_losses) / len(val_losses)
    print(f'Epoch {epoch + 1}/{num_epochs}, Validation Loss: {avg_val_loss}')

# Save the fine-tuned model
model.save_pretrained("fine_tuned_bert_model")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2614: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In t

Epoch 1/10, Validation Loss: 0.04336779675099487
Epoch 2/10, Validation Loss: 0.014970916026618843
Epoch 3/10, Validation Loss: 0.018481530153247362
Epoch 4/10, Validation Loss: 0.08556239033236589
Epoch 5/10, Validation Loss: 0.04473436697427293
Epoch 6/10, Validation Loss: 0.045099862596115974
Epoch 7/10, Validation Loss: 0.04660329660777386
Epoch 8/10, Validation Loss: 0.04679747709237477
Epoch 9/10, Validation Loss: 0.04725598137769802
Epoch 10/10, Validation Loss: 0.049738177483816925


In [16]:
test_df

,text,label
2276,"In recent years, there has been a growing tren...",1
2277,"Dear Principal,\n\nI am writing to express my ...",1
2278,"Hey, y'all! For this essay, I had to research...",1
2279,The Electoral College was a system thought up ...,0
2280,Biking during the summer can be a fun and enjo...,1
...,...,...
2673,"State Senator, The Electoral College is not a ...",0
2674,"Dear State Senator of Florida, I believe that ...",0
2675,"Dear State Senator, The Electoral College is a...",0
2676,There are many ways of limiting car usage. Som...,0


In [17]:
# Load the fine-tuned BERT model
model = BertForSequenceClassification.from_pretrained("fine_tuned_bert_model")
model.to('cuda')
# Load the test dataset
# Replace 'test_dataset.csv' with the path to your test set CSV file
# test_df = pd.read_csv('test_dataset.csv')

# Tokenize the test data
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = str(self.data.iloc[idx]['text'])
        label = int(self.data.iloc[idx]['label'])

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Create the test dataset and dataloader
test_dataset = TextDataset(test_df, tokenizer, max_length=128)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Evaluation
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['label'].to('cuda')

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        _, predicted_class = torch.max(logits, dim=1)

        predictions.extend(predicted_class.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# Calculate accuracy and other metrics
accuracy = accuracy_score(true_labels, predictions)
classification_report_str = classification_report(true_labels, predictions)

print(f"Test Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report_str)

Test Accuracy: 1.0000
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       209
           1       1.00      1.00      1.00       193

    accuracy                           1.00       402
   macro avg       1.00      1.00      1.00       402
weighted avg       1.00      1.00      1.00       402

